# ISS pixel-based decoding with Starfish

This notebook performs genuine pixel-based decoding with Starfish's `PixelSpotDecoder`. Every pixel is matched to the closest SpaceTx codeword before adjacent pixels assigned to the same target are merged into transcript-like connected components.

This differs from the existing `dense=True` workflow, which still detects discrete spots before decoding. Pixel decoding can help when spots overlap, but standard one-hot ISS codebooks are less error-robust than codebooks designed for pixel decoding, so conservative magnitude, distance, and area thresholds are important.

Pixel decoding is CPU- and RAM-intensive. The current preprocessing default of 6000×6000 pixels per retiled image is not a safe starting point: a six-round, five-channel raw trace array is already about 4.3 GB before Starfish's indexed and normalized copies. Start with one region made from 512×512 or 1024×1024 tiles and monitor memory before scaling up.

## We start by importing the necessary modules

In [ ]:
import ISS_decoding.SpaceTx_format as STX
import ISS_decoding.decoding as DEC
import pandas as pd
from pathlib import Path

## Format the images to SpaceTx format

The first thing to do before we can start the actual decoding is to transform our images (the resliced tiles) to the SpaceTx format.

To read more about the SpaceTx format, read the following: https://github.com/spacetx/sptx-format

### Parameters

`input_dir` = type: `str`. Path to the parent directory containing the preprocessed region folders (e.g., `/R1/`, `/R2/`, …).  
These region folders are automatically generated by the preprocessing module.  
The `input_dir` is referenced throughout this notebook.

`codebook_csv` = type: `str`. This is a file that associates a unique color sequence across ISS cycles to each gene. This file is a comma separated file with no header, in which the first column contains the gene name, while  columns 2 to 6 (in case of a 6 cycle experiment) contain numbers representing the expected positive DO_decorator in each cycle for that gene. 

`regions_to_process` = type:`list[int]` | None, default:None. A list of 1-based region indices defining which regions should be processed. If None → all detected regions are processed.

`output_dir_prefix` = type: `str` | None, default: `None`.  
Optional base directory where SpaceTx outputs should be written.

- If `output_dir_prefix` is `None`, SpaceTx outputs are written **inside each region directory under `input_dir`**, i.e.  
  `input_dir/R#/decoding/1_SpaceTX_format/`.

- If `output_dir_prefix` is set, SpaceTx outputs are written under:  
  `output_dir_prefix/R#/decoding/1_SpaceTX_format/`.

`pixel_to_um`  = type: `float`. 
Physical size of one pixel in microns (µm per pixel). This value determines the units of the spatial coordinates written to the experiment metadata and decoding outputs:

- `pixel_to_um = 1.0` (default) → coordinates are pixel-based.

- `pixel_to_um = 0.1625` (or microscope-specific value) → coordinates are in microns.


`channels` = type: `list`. The channels, in the order they were acquired in the microscope. Default = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"]

`DO_decorators` =  type: `list` **This point can be tricky to understand.** This shows how you associate the numbers in the codebook (ie 1,2,3,4) to a specific list of colors (DO_decorator) . In our lab, 1,2,3,4,5 correspond to ["AF750", "AF488", "Cy3", "Cy5", "At425"]. 

Default = ["AF750", "AF488", "Cy3", "Cy5", "At425"]. Users who follow our barcode design and readout schemes should not change this.

`nuclei_channel`  = type: `str`. This is the name of the channel that corresponds to your nuclei stained image. Default =  "DAPI".

`CARE` = type: `bool`.
If set to `True`, the function will use CARE-denoised retiled images as input.

- `CARE = False` (default):
Tiles are read from
`/preprocessing/<CycleX>/4_retiled/`

- `CARE = True`:
Tiles are read from
`/preprocessing/<CycleX>/4_retiled/CARE/`
This folder must already exist and should contain the outputs produced by the ISS_CARE preprocessing step.
This parameter does not run CARE itself. It only controls which image directory is used as input when building the experiment and codebook files.
Users should set `CARE = True` only after CARE denoising has been successfully completed for the corresponding regions and cycles.

In [ ]:
input_dir = '/path/to/regions/'
codebook_csv = '/path/to/codebook/'

In [ ]:
STX.make_spacetx_format(
    input_dir,
    codebook_csv,
    regions_to_process = None,    # or for example [2]
    output_dir_prefix = None,     # or '/path/to/preferred/output/dir'
    pixel_to_um = 1,
    channels = ["AF750", "Cy5", "Cy3", "AF488", "DAPI", "At425"],
    DO_decorators = ["AF750", "AF488", "Cy3", "Cy5", "At425"],
    nuclei_channel = "DAPI",
    CARE = True                  # True if using CARE denoised images
    )


## Decode the SpaceTx formatted data

`decode_mode='PIXEL'` uses Starfish's metric decoder on every pixel and then applies connected-component labeling. It does not use BlobDetector, Spotiflow, `int_threshold`, or `sigma_vals`.

### Pixel-decoding parameters

- `metric`: distance metric used to compare normalized pixel traces with codewords. Start with `'euclidean'`.
- `distance_threshold`: maximum normalized distance from the nearest codeword. Smaller values are stricter; `0.5` is a reasonable pilot value, not a universal cutoff.
- `magnitude_threshold`: minimum unnormalized pixel-trace magnitude. Increase it to reject dim background pixels.
- `min_area`, `max_area`: allowed connected-component area in pixels. Components outside the interval remain in the raw table with `passes_thresholds=False`.
- `norm_order`: norm used before metric matching; Starfish defaults to L2.
- `n_processes`: worker count for connected-component attribute measurement. The nearest-code search itself is not GPU-accelerated and is not controlled by this setting.

The output includes `pixel_distance`, `pixel_area`, `passes_thresholds`, the effective thresholds, standard coordinates, and the decoded `target`. Tune the three thresholds on a small representative region before a full run.

In [ ]:
DEC.process_experiment(
    input_dir,
    regions_to_process = [1],              # begin with one representative region
    output_dir_prefix = None,              # or '/path/to/preferred/output/dir'
    register = False,
    register_dapi = False,
    masking_radius = 7,
    normalization_method = 'MH',
    decode_mode = 'PIXEL',
    dense = False,                         # PIXEL is separate from dense spot decoding
    pixel_kwargs = {
        'metric': 'euclidean',
        'distance_threshold': 0.5,
        'magnitude_threshold': 0.1,
        'min_area': 2,
        'max_area': 100,
        'norm_order': 2,
        'n_processes': 1,
    },
    )

# Explore the decoded data

The decoding step writes a canonical region-level Parquet table plus a CSV compatibility copy. The table retains all connected components, including components outside the configured area interval; use `passes_thresholds` when you are ready to apply an experiment-specific filter.

#### Processing Reads for Individual Regions

We will now load one decoded region (`R1`, `R2`, etc.). Pixel-decoding outputs are stored separately under `2_decoded_pixel`.

In [ ]:
region = 'R1'

read_file = (
    Path(input_dir)
    / region
    / 'decoding'
    / '2_decoded_pixel'
    / f"{region}_decoded_pixel.parquet"
)

reads = pd.read_parquet(read_file)

The number of extracted raw reads for this region is:

In [ ]:
len(reads)